# 🎬 CorridorKey 2.0 — Video Background Remover

Upload a video and get a **ProRes 4444 `.mov` with an alpha channel**.

**Quick start:** click **Runtime → Run all**, then choose your video when the file picker appears. The defaults process the **entire clip** with no hint and write one transparent video.

**What you need:** a Colab GPU runtime (**Runtime → Change runtime type → Hardware accelerator: GPU**). A free T4 can run via a local float16 compatibility patch (experimental and slow); an A100/L4 is recommended. The first run installs PyTorch and downloads ~5 GB of model weights (about 10–15 minutes). Later runs in the same session skip that.

**What you get:** a transparent ProRes 4444 `.mov` (the edit/compositing deliverable), an 8-second browser-playable preview over a checkerboard, and a first-frame PNG. Original audio is muxed back into a separate `_cutout_with_audio.mov` when possible.

> **Long clips:** the whole video is processed by default. On a T4 this can take a long time, and a Colab disconnect erases `/content`. Download results before the session ends. The advanced section can limit frames or process a 720p copy. Colab's upload widget is memory-hungry for multi-GB files; upload a shorter section if needed.

<details>
<summary><b>License (short version)</b></summary>

CorridorKey 2.0 was designed and trained by Zachary Harris in collaboration with Niko and Corridor Digital. You may use it and the outputs for personal or commercial work. You may not sell the tool, its code, or its model, and you may not offer it as a paid API/inference service without permission. If you build on it, your release must stay free, keep the same license, and keep the CorridorKey name. Organizations with more than US $20M annual revenue need a plugin seat per user. Full terms: [LICENSE](https://github.com/nikopueringer/CorridorKey2.0/blob/master/LICENSE). Donationware and pay-what-you-want is fine.

</details>


In [ ]:
#@title 1. Check GPU { display-mode: "form" }
import subprocess

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print(smi.stdout.strip() or "nvidia-smi did not report a GPU.")

try:
    import torch
except Exception as exc:
    raise SystemExit(f"PyTorch could not be imported: {exc}")

if not torch.cuda.is_available():
    raise SystemExit(
        "No CUDA GPU is available.\n"
        "In Colab: Runtime > Change runtime type > Hardware accelerator > GPU, then re-run this cell."
    )

props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9
print(f"GPU: {props.name} | VRAM: {vram_gb:.1f} GB | compute capability: {props.major}.{props.minor}")
print(f"PyTorch (before install): {torch.__version__}")
if props.major < 7:
    raise SystemExit(
        "This GPU is too old for the CUDA 12.8 PyTorch build used here "
        "(minimum compute capability 7.0). Use a T4, A100, L4, or newer."
    )
if props.major < 8:
    print(
        "\nNOTE: pre-Ampere GPU (T4/Turing) detected. T4 has no native bfloat16, so the setup cell "
        "applies a local float16 compatibility patch. This is experimental; an A100/L4 is recommended "
        "for production and long clips."
    )
if vram_gb < 7:
    print("\nWARNING: less than 7 GB of VRAM. Low-VRAM mode will be used automatically; 4K video may not fit.")


In [ ]:
#@title 2. Upload your video { display-mode: "form" }
import gc
import json
import re
import shutil
import subprocess
from pathlib import Path

from google.colab import files
from IPython.display import display
from PIL import Image

if shutil.which("ffmpeg") is None:
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)

uploaded = files.upload()
if not uploaded:
    raise SystemExit("No file was selected. Re-run this cell and choose a video.")
if len(uploaded) > 1:
    print("You selected more than one file; using the first.")

raw_name = next(iter(uploaded))
raw_path = Path(raw_name)
if not raw_path.is_file():
    raw_path = Path("/content") / raw_name
    raw_path.write_bytes(uploaded[raw_name])

stem = Path(raw_name).stem
safe = re.sub(r"[^A-Za-z0-9._-]+", "_", stem).strip("._") or "clip"
VIDEO_NAME = safe
work = Path("/content/work") / VIDEO_NAME
work.mkdir(parents=True, exist_ok=True)

suffix = Path(raw_name).suffix.lower() or ".mp4"
VIDEO_ORIGINAL = work / f"{VIDEO_NAME}{suffix}"
if raw_path.resolve() != VIDEO_ORIGINAL.resolve():
    shutil.move(str(raw_path), str(VIDEO_ORIGINAL))
else:
    VIDEO_ORIGINAL = VIDEO_ORIGINAL.resolve()


def probe_video(path):
    cmd = [
        "ffprobe", "-v", "error", "-select_streams", "v:0",
        "-show_streams", "-show_format", "-of", "json", str(path),
    ]
    data = json.loads(subprocess.check_output(cmd, text=True))
    if not data.get("streams"):
        raise SystemExit(f"ffprobe found no video stream in {path}")
    stream = data["streams"][0]
    rate = stream.get("avg_frame_rate") or stream.get("r_frame_rate") or "0/1"
    try:
        num, den = rate.split("/")
        fps = float(num) / float(den) if float(den) else 0.0
    except Exception:
        fps = 0.0
    if fps <= 0:
        fps = 24.0
    duration = float(stream.get("duration") or data.get("format", {}).get("duration") or 0.0)
    frames = int(stream.get("nb_frames") or 0)
    if frames <= 0 and fps > 0 and duration > 0:
        frames = int(round(duration * fps))
    rotation = 0
    for item in stream.get("side_data_list", []):
        if "rotation" in item:
            try:
                rotation = int(float(item["rotation"]))
            except Exception:
                pass
    if not rotation:
        try:
            rotation = int(float(stream.get("tags", {}).get("rotate", 0)))
        except Exception:
            pass
    return {
        "width": int(stream["width"]),
        "height": int(stream["height"]),
        "fps": fps,
        "duration": duration,
        "frames": frames,
        "rotation": rotation,
    }


info = probe_video(VIDEO_ORIGINAL)
print("Original:", VIDEO_ORIGINAL)
print(f"{info['width']}x{info['height']} | {info['fps']:.3f} fps | {info['duration']:.2f} s | ~{info['frames']} frames")

if info["rotation"]:
    print("Rotation metadata detected:", info["rotation"], "degrees. Creating an auto-rotated copy...")
    norm_dir = work / "normalized"
    norm_dir.mkdir(parents=True, exist_ok=True)
    normalized = norm_dir / f"{VIDEO_NAME}.mp4"
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", str(VIDEO_ORIGINAL),
         "-c:v", "libx264", "-crf", "16", "-preset", "veryfast",
         "-pix_fmt", "yuv420p", "-an", str(normalized)],
        check=True,
    )
    VIDEO_SOURCE = normalized
    info = probe_video(VIDEO_SOURCE)
    print(f"Normalized: {info['width']}x{info['height']} | {info['fps']:.3f} fps")
else:
    VIDEO_SOURCE = VIDEO_ORIGINAL

thumb = work / "thumbnail.png"
subprocess.run(
    ["ffmpeg", "-y", "-loglevel", "error", "-i", str(VIDEO_SOURCE),
     "-frames:v", "1", str(thumb)],
    check=True,
)
thumb_img = Image.open(thumb)
thumb_img.thumbnail((1024, 1024), Image.LANCZOS)
display(thumb_img)

del uploaded
gc.collect()


In [ ]:
#@title 3. Install CorridorKey 2.0 (first run ~10–15 min) { display-mode: "form" }
#@markdown Clones the repo, installs the tested CUDA PyTorch + dependencies, checks ffmpeg, and downloads the ~5 GB weights.
#@markdown On T4/Turing GPUs it also applies a local float16 compatibility patch (experimental), because those GPUs lack native bfloat16.
#@markdown Re-running is safe and skips finished steps.
force_reinstall = False #@param {type:"boolean"}

import json
import shutil
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/CorridorKey2.0")
REPO_URL = "https://github.com/nikopueringer/CorridorKey2.0.git"
MARKER = Path("/content/.corridorkey_setup_v1.json")
TORCH_VERSION = "2.9.1"
TV_VERSION = "0.24.1"
PATCH_FILES = ("corridorkey/runtime.py", "corridorkey/video_io.py")
PATCH_MARKER = REPO_DIR / ".t4_fp16_patch"


def run(cmd, cwd=None):
    print("$", " ".join(str(c) for c in cmd), flush=True)
    subprocess.run([str(c) for c in cmd], cwd=str(cwd) if cwd else None, check=True)


def torch_state():
    code = (
        "import torch, torchvision; "
        "print(torch.__version__, torchvision.__version__, int(torch.cuda.is_available()))"
    )
    try:
        out = subprocess.check_output([sys.executable, "-c", code], text=True).strip().split()
        if len(out) >= 3:
            return out[0], out[1], out[2] == "1"
    except Exception:
        pass
    return None, None, False


# Restore upstream files if a previous T4 patch is present, so git can update cleanly.
if (REPO_DIR / "infer.py").is_file() and PATCH_MARKER.is_file():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "checkout", "--", *PATCH_FILES],
        capture_output=True, text=True,
    )
    try:
        PATCH_MARKER.unlink()
    except FileNotFoundError:
        pass
    print("Restored upstream files before updating.", flush=True)

# 1) Repository
if not (REPO_DIR / "infer.py").is_file():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    try:
        run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
    except subprocess.CalledProcessError:
        print("Could not update the existing checkout; continuing with it.", flush=True)

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
print("CorridorKey commit:", commit, flush=True)

marker = {}
if MARKER.is_file():
    try:
        marker = json.loads(MARKER.read_text())
    except Exception:
        marker = {}
same_setup = (not force_reinstall and marker.get("commit") == commit)

# 2) PyTorch
torch_v, tv_v, cuda_ok = torch_state()
need_torch = (
    force_reinstall
    or torch_v is None
    or not torch_v.startswith(TORCH_VERSION)
    or tv_v is None
    or not tv_v.startswith(TV_VERSION)
    or not cuda_ok
)
if need_torch:
    print(f"Installing PyTorch {TORCH_VERSION} + torchvision {TV_VERSION} (CUDA 12.8)...", flush=True)
    run([
        sys.executable, "-m", "pip", "install", "-q",
        f"torch=={TORCH_VERSION}", f"torchvision=={TV_VERSION}",
        "--index-url", "https://download.pytorch.org/whl/cu128",
    ])
else:
    print(f"PyTorch {torch_v} + torchvision {tv_v} already installed; skipping.", flush=True)

# 2b) T4/Turing compatibility: bfloat16 is not native on pre-Ampere GPUs.
try:
    cap_major = int(subprocess.check_output(
        [sys.executable, "-c", "import torch; print(torch.cuda.get_device_capability(0)[0])"],
        text=True,
    ).strip())
except Exception:
    cap_major = 8

if cap_major < 8:
    patched = []
    for rel in PATCH_FILES:
        p = REPO_DIR / rel
        text = p.read_text(encoding="utf-8")
        if "torch.bfloat16" in text:
            p.write_text(text.replace("torch.bfloat16", "torch.float16"), encoding="utf-8")
            patched.append(rel)
    PATCH_MARKER.write_text("fp16\n", encoding="utf-8")
    print("T4/Turing detected: applied the local float16 compatibility patch.", flush=True)
    print("Patched:", ", ".join(patched) if patched else "already patched", flush=True)
else:
    print("Ampere+ GPU detected: using the native bfloat16 path.", flush=True)

# 3) Python dependencies
deps_ok = subprocess.run(
    [sys.executable, "-c", "import cv2, safetensors, transformers, timm, einops, scipy, utils3d"],
    capture_output=True,
).returncode == 0
if force_reinstall or not same_setup or not deps_ok:
    print("Installing Python dependencies...", flush=True)
    run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")])
else:
    print("Python dependencies already installed for this commit; skipping.", flush=True)

# 4) ffmpeg + ProRes encoder
if shutil.which("ffmpeg") is None:
    run(["apt-get", "-qq", "install", "-y", "ffmpeg"])
encoders = subprocess.check_output(
    ["ffmpeg", "-hide_banner", "-encoders"], text=True, errors="ignore"
)
if "prores_ks" not in encoders:
    raise RuntimeError(
        "This ffmpeg build has no prores_ks encoder. The transparent .mov output cannot be written. "
        "Re-run: apt-get install -y ffmpeg"
    )
print("ffmpeg + prores_ks OK", flush=True)

# 5) Weights
required = [
    REPO_DIR / "weights/corridorkey_v2.safetensors",
    REPO_DIR / "weights/foundation_stats.safetensors",
    REPO_DIR / "vfms/moge/model.pt",
]
cradio_dir = REPO_DIR / "vfms/cradio/C-RADIOv4-SO400M"
rvm_dir = REPO_DIR / "vfms/rvm"
missing = [p for p in required if not p.is_file()]
if not cradio_dir.is_dir() or not any(cradio_dir.iterdir()):
    missing.append(cradio_dir)
if not rvm_dir.is_dir() or not any(rvm_dir.glob("*.npz")):
    missing.append(rvm_dir)

if force_reinstall or missing:
    print("Downloading model weights (~5 GB, resumable)...", flush=True)
    run([sys.executable, str(REPO_DIR / "scripts/download_weights.py")], cwd=REPO_DIR)
else:
    print("Model weights already present; skipping.", flush=True)

# 6) Done
MARKER.write_text(json.dumps({
    "commit": commit,
    "torch": TORCH_VERSION,
    "torchvision": TV_VERSION,
}))
print("\nSetup complete.", flush=True)
print("Repo:", REPO_DIR, flush=True)
print(
    "If Colab shows a 'restart runtime' banner, you can ignore it: "
    "inference runs in a fresh subprocess.",
    flush=True,
)


In [ ]:
#@title ⚙️ Advanced settings (optional) { display-mode: "form" }
#@markdown The defaults process the entire clip and write one transparent ProRes 4444 .mov.
save_alpha = False #@param {type:"boolean"}
#@markdown Also save the black/white matte.
save_fg = False #@param {type:"boolean"}
#@markdown Also save the raw foreground RGB.
save_checker = False #@param {type:"boolean"}
#@markdown Also save a full-resolution foreground-over-checker preview.
max_frames = 0 #@param {type:"integer"}
#@markdown 0 = entire video. Otherwise process this many frames (see start_mode).
start_mode = "begin" #@param ["begin", "middle", "random_middle"]
#@markdown Where to start when max_frames is set. Only used if max_frames > 0.
seed = 0 #@param {type:"integer"}
#@markdown Seed for random_middle.
hint_quality = 0.95 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
#@markdown Trust in the optional hint: 0.95 = clean matte, 0.75 = rough, <=0.5 = loose.
carry_hint = True #@param {type:"boolean"}
#@markdown Reuse the model's own overlapping-window predictions as a continuing hint.
carry_hint_quality = 0.95 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
#@markdown Trust in carried hints (when carry_hint is on).
despill_strength = 0.5 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
#@markdown Green/blue spill removal in the cutout. 0 = none, 1 = full.
memory_mode = "Auto (recommended)" #@param ["Auto (recommended)", "Force low VRAM", "Manual Hann window"]
#@markdown Auto picks the best window from free GPU memory. Force low VRAM saves memory. Manual exposes the window controls below.
hann_chunk = 8 #@param {type:"integer"}
#@markdown Manual mode only: frames per stage-1 window. 8 = lower memory, 80 = max context.
hann_stride = 4 #@param {type:"integer"}
#@markdown Manual mode only: frames between windows. 0 = half of hann_chunk.
normalize_video = False #@param {type:"boolean"}
#@markdown Re-encode the input to H.264 before inference (fixes some codecs/containers).
downscale_720p = False #@param {type:"boolean"}
#@markdown Process a 720p copy for speed. The output is 720p.
keep_audio = True #@param {type:"boolean"}
#@markdown Mux the original audio into the transparent .mov when possible.
cutout_linear = False #@param {type:"boolean"}
#@markdown Export straight scene-linear RGB instead of sRGB. For linear VFX compositing only.


In [ ]:
#@title 4. Optional: download the first frame (for painting a matte) { display-mode: "form" }
#@markdown Use this if you want to paint a black/white matte for the first frame, then upload it in the next cell.
EXTRACT_FIRST_FRAME = False #@param {type:"boolean"}

import subprocess
from pathlib import Path

from IPython.display import FileLink, display
from PIL import Image

if EXTRACT_FIRST_FRAME:
    if "VIDEO_SOURCE" not in globals():
        raise SystemExit("Run the video upload cell first.")
    frame_path = Path("/content/work") / VIDEO_NAME / "first_frame.png"
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", str(VIDEO_SOURCE),
         "-frames:v", "1", str(frame_path)],
        check=True,
    )
    print("First frame:", frame_path)
    frame_img = Image.open(frame_path)
    frame_img.thumbnail((1024, 1024), Image.LANCZOS)
    display(frame_img)
    display(FileLink(str(frame_path), result_html_prefix="Download first frame: "))
else:
    print("Skipped. Turn this on if you need a reference frame for a matte.")


In [ ]:
#@title 5. Optional: upload hints (better tracking/control) { display-mode: "form" }
#@markdown A matte is black/white: **white = subject**, **black = background**. RGBA hints use the alpha channel.
#@markdown A first-frame matte is usually the best practical mode. A full hint video is the most controlled mode.
ADD_FIRST_FRAME_HINT = False #@param {type:"boolean"}
ADD_FULL_HINT_VIDEO = False #@param {type:"boolean"}

import shutil
from pathlib import Path

from google.colab import files
from IPython.display import display
from PIL import Image

if "VIDEO_NAME" not in globals():
    raise SystemExit("Run the video upload cell first.")

HINT_FIRST_FRAME_SOURCE = None
HINT_VIDEO_SOURCE = None

if ADD_FIRST_FRAME_HINT:
    uploaded_hint = files.upload()
    if uploaded_hint:
        name = next(iter(uploaded_hint))
        p = Path(name)
        if not p.is_file():
            p = Path("/content") / name
            p.write_bytes(uploaded_hint[name])
        HINT_FIRST_FRAME_SOURCE = Path("/content/work") / VIDEO_NAME / "hint_first_source.png"
        try:
            im = Image.open(p)
        except Exception as exc:
            raise SystemExit(f"Could not read the first-frame matte: {exc}")
        im.save(HINT_FIRST_FRAME_SOURCE)
        print("First-frame matte:", HINT_FIRST_FRAME_SOURCE, im.size)
        thumb = im.copy()
        thumb.thumbnail((1024, 1024), Image.LANCZOS)
        display(thumb)
    else:
        print("No first-frame matte selected.")
else:
    print("No first-frame matte.")

if ADD_FULL_HINT_VIDEO:
    uploaded_hint = files.upload()
    if uploaded_hint:
        name = next(iter(uploaded_hint))
        p = Path(name)
        if not p.is_file():
            p = Path("/content") / name
            p.write_bytes(uploaded_hint[name])
        ext = p.suffix.lower() or ".mp4"
        HINT_VIDEO_SOURCE = Path("/content/work") / VIDEO_NAME / f"hint_video{ext}"
        if p.resolve() != HINT_VIDEO_SOURCE.resolve():
            shutil.move(str(p), str(HINT_VIDEO_SOURCE))
        else:
            HINT_VIDEO_SOURCE = HINT_VIDEO_SOURCE.resolve()
        print("Full hint video:", HINT_VIDEO_SOURCE)
    else:
        print("No full hint video selected.")
else:
    print("No full hint video.")


In [ ]:
#@title 6. Remove the background { display-mode: "form" }
import json
import shutil
import subprocess
import sys
import time
from pathlib import Path

from PIL import Image

if "VIDEO_SOURCE" not in globals() or not Path(VIDEO_SOURCE).is_file():
    raise SystemExit("Run the video upload cell first.")

save_alpha = bool(globals().get("save_alpha", False))
save_fg = bool(globals().get("save_fg", False))
save_checker = bool(globals().get("save_checker", False))
max_frames = max(0, int(globals().get("max_frames", 0)))
start_mode = str(globals().get("start_mode", "begin"))
seed = int(globals().get("seed", 0))
hint_quality = float(globals().get("hint_quality", 0.95))
carry_hint = bool(globals().get("carry_hint", True))
carry_hint_quality = float(globals().get("carry_hint_quality", 0.95))
despill_strength = float(globals().get("despill_strength", 0.5))
memory_mode = str(globals().get("memory_mode", "Auto (recommended)"))
hann_chunk = int(globals().get("hann_chunk", 8))
hann_stride = int(globals().get("hann_stride", 4))
normalize_video = bool(globals().get("normalize_video", False))
downscale_720p = bool(globals().get("downscale_720p", False))
cutout_linear = bool(globals().get("cutout_linear", False))

REPO_DIR = Path(globals().get("REPO_DIR", "/content/CorridorKey2.0"))
infer_py = REPO_DIR / "infer.py"
if not infer_py.is_file():
    raise SystemExit("CorridorKey is not installed. Run the setup cell above.")

work = Path("/content/work") / VIDEO_NAME
work.mkdir(parents=True, exist_ok=True)
video_in = Path(VIDEO_SOURCE)


def probe(path):
    cmd = [
        "ffprobe", "-v", "error", "-select_streams", "v:0",
        "-show_entries", "stream=width,height,avg_frame_rate,r_frame_rate,nb_frames:format=duration",
        "-of", "json", str(path),
    ]
    data = json.loads(subprocess.check_output(cmd, text=True))
    stream = data["streams"][0]
    rate = stream.get("avg_frame_rate") or stream.get("r_frame_rate") or "0/1"
    try:
        num, den = rate.split("/")
        fps = float(num) / float(den) if float(den) else 0.0
    except Exception:
        fps = 0.0
    if fps <= 0:
        fps = 24.0
    duration = float(data.get("format", {}).get("duration") or 0.0)
    frames = int(stream.get("nb_frames") or 0)
    if frames <= 0 and fps > 0 and duration > 0:
        frames = int(round(duration * fps))
    return int(stream["width"]), int(stream["height"]), fps, frames


if normalize_video or downscale_720p:
    prep_dir = work / "prepared"
    prep_dir.mkdir(parents=True, exist_ok=True)
    prepared = prep_dir / f"{VIDEO_NAME}.mp4"
    vf = ["-vf", "scale=-2:720"] if downscale_720p else []
    print("Preparing input...", flush=True)
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", str(video_in), *vf,
         "-c:v", "libx264", "-crf", "18", "-preset", "veryfast",
         "-pix_fmt", "yuv420p", "-an", str(prepared)],
        check=True,
    )
    video_in = prepared
    print("Prepared:", prepared)

pw, ph, pfps, pframes = probe(video_in)
print(f"Processed input: {pw}x{ph} | {pfps:.3f} fps | ~{pframes} frames")

hint_first = None
hint_first_src = globals().get("HINT_FIRST_FRAME_SOURCE")
if hint_first_src and Path(hint_first_src).is_file():
    hint = Image.open(hint_first_src)
    hint = hint.convert("RGBA") if hint.mode in ("RGBA", "LA") else hint.convert("L")
    if hint.size != (pw, ph):
        print(f"Resizing first-frame matte from {hint.size} to {(pw, ph)}.")
        hint = hint.resize((pw, ph), Image.LANCZOS)
    hint_first = work / "hint_first_frame.png"
    hint.save(hint_first)
    print("Using first-frame matte:", hint_first)

hint_video = None
hint_vid_src = globals().get("HINT_VIDEO_SOURCE")
if hint_vid_src and Path(hint_vid_src).is_file():
    hw, hh, _hfps, hframes = probe(hint_vid_src)
    if (hw, hh) != (pw, ph):
        raise SystemExit(
            f"Full hint video is {hw}x{hh}, but the processed video is {pw}x{ph}. "
            "Re-export the hint at the processed resolution, or disable downscale_720p/normalize_video."
        )
    required = pframes
    if max_frames > 0 and start_mode == "begin":
        required = min(max_frames, pframes)
    if hframes and required and hframes < required:
        raise SystemExit(f"Full hint video has {hframes} frames; at least {required} are required.")
    hint_video = Path(hint_vid_src)
    print("Using full hint video:", hint_video)

outputs = ["cutout"]
if save_alpha:
    outputs.append("alpha")
if save_fg:
    outputs.append("fg")
if save_checker:
    outputs.append("checker")

out_dir = Path("/content/outputs") / VIDEO_NAME
if out_dir.exists():
    shutil.rmtree(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, str(infer_py),
    "--input", str(video_in),
    "--output_dir", str(out_dir),
    "--outputs", *outputs,
    "--hint_quality", str(hint_quality),
    "--carry_hint_quality", str(carry_hint_quality),
    "--despill_strength", str(despill_strength),
]
if not carry_hint:
    cmd.append("--no_carry_hint")
if hint_first is not None:
    cmd += ["--hint_first_frame", str(hint_first)]
if hint_video is not None:
    cmd += ["--hint_video", str(hint_video)]
if max_frames > 0:
    cmd += ["--num_frames", str(max_frames), "--start_mode", start_mode, "--seed", str(seed)]
if memory_mode == "Force low VRAM":
    cmd.append("--low_vram")
elif memory_mode == "Manual Hann window":
    cmd += ["--hann_chunk", str(max(1, hann_chunk))]
    if hann_stride and hann_stride > 0:
        cmd += ["--hann_stride", str(hann_stride)]
if cutout_linear:
    cmd.append("--cutout_linear")

print("\nCommand:")
print(" ".join(cmd))
print("Output directory:", out_dir)
if memory_mode == "Auto (recommended)":
    print("Memory: auto (chosen from free GPU memory at run time)")
print()
start = time.time()
try:
    subprocess.run(cmd, check=True)
except subprocess.CalledProcessError as exc:
    print("\nInference failed with exit code", exc.returncode)
    print("Common fixes:")
    print("- CUDA out of memory: set memory_mode to 'Force low VRAM', or 'Manual Hann window' with hann_chunk=8/hann_stride=4 (or 4/2), then re-run.")
    print("- Still out of memory: enable downscale_720p, reduce max_frames, or use a GPU with more VRAM.")
    print("- Missing weights: re-run the setup cell.")
    print("- No GPU: Runtime > Change runtime type > GPU.")
    raise
print(f"\nDone in {time.time() - start:.1f} s.")
print("Outputs are in:", out_dir)

OUT_DIR = out_dir
CLIP_NAME = VIDEO_NAME
PROCESSED_VIDEO = video_in


In [ ]:
#@title 7. Preview & download { display-mode: "form" }
import json
import subprocess
from pathlib import Path

import numpy as np
from IPython.display import FileLink, Image as IPImage, Video, display
from PIL import Image

if "OUT_DIR" not in globals():
    raise SystemExit("Run the inference cell first.")

out_dir = Path(OUT_DIR)
clip = CLIP_NAME
keep_audio = bool(globals().get("keep_audio", True))
max_frames = max(0, int(globals().get("max_frames", 0)))
start_mode = str(globals().get("start_mode", "begin"))


def human(n):
    n = float(n)
    for unit in ("B", "KB", "MB", "GB"):
        if n < 1024 or unit == "GB":
            return f"{n:.1f} {unit}"
        n /= 1024


def checker_rgba(width, height, tile=32):
    yy, xx = np.indices((height, width))
    mask = (((xx // tile) + (yy // tile)) % 2).astype(np.uint8)
    gray = np.where(mask == 1, 210, 160).astype(np.uint8)
    rgb = np.repeat(gray[:, :, None], 3, axis=2)
    return Image.fromarray(rgb, "RGB").convert("RGBA")


def show_image(path, max_side=1024):
    img = Image.open(path)
    img.thumbnail((max_side, max_side), Image.LANCZOS)
    display(img)


def ffprobe_video(path):
    cmd = [
        "ffprobe", "-v", "error", "-select_streams", "v:0",
        "-show_entries", "stream=width,height,avg_frame_rate,r_frame_rate",
        "-of", "json", str(path),
    ]
    data = json.loads(subprocess.check_output(cmd, text=True))
    stream = data["streams"][0]
    rate = stream.get("avg_frame_rate") or stream.get("r_frame_rate") or "24/1"
    try:
        num, den = rate.split("/")
        fps = float(num) / float(den) if float(den) else 24.0
    except Exception:
        fps = 24.0
    if fps <= 0:
        fps = 24.0
    return int(stream["width"]), int(stream["height"]), fps


def extract_rgba(video_path, png_path):
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", str(video_path),
         "-frames:v", "1", "-pix_fmt", "rgba", str(png_path)],
        check=True,
    )


cutout = out_dir / f"{clip}_cutout.mov"
still = out_dir / f"{clip}_frame0_checker.png"
preview = out_dir / f"{clip}_preview_checker.mp4"

if cutout.is_file():
    rgba_png = out_dir / f"{clip}_frame0_rgba.png"
    extract_rgba(cutout, rgba_png)
    rgba = Image.open(rgba_png).convert("RGBA")
    bg = checker_rgba(rgba.width, rgba.height)
    comp = Image.alpha_composite(bg, rgba).convert("RGB")
    comp.save(still)
    print("Frame 0 (result over a checkerboard):")
    show_image(still, 1024)
    print("Alpha range:", rgba.getchannel("A").getextrema())

    width, height, fps = ffprobe_video(cutout)
    if width > 1280:
        pw = 1280
        ph = max(2, int(round(height * pw / width / 2)) * 2)
    else:
        pw = width if width % 2 == 0 else width - 1
        ph = height if height % 2 == 0 else height - 1
    checker_path = out_dir / "preview_checker.png"
    checker_rgba(pw, ph).convert("RGB").save(checker_path)

    preview_cmd = [
        "ffmpeg", "-y", "-loglevel", "error",
        "-loop", "1", "-framerate", f"{fps:.6f}", "-i", str(checker_path),
        "-i", str(cutout),
        "-filter_complex",
        f"[1:v]scale={pw}:{ph}:flags=lanczos[fg];[0:v][fg]overlay=shortest=1:format=auto,format=yuv420p[v]",
        "-map", "[v]", "-t", "8", "-r", f"{fps:.6f}", "-an",
        "-c:v", "libx264", "-crf", "20", "-preset", "veryfast",
        "-movflags", "+faststart", str(preview),
    ]
    preview_ok = False
    try:
        subprocess.run(preview_cmd, check=True)
        preview_ok = True
        print("\nPreview (first 8 seconds, over a checkerboard):")
        display(Video(str(preview), embed=True, width=640))
    except subprocess.CalledProcessError as exc:
        print("Preview transcode failed:", exc)
        print("The full output is still available below.")

    result = cutout
    if keep_audio and "VIDEO_ORIGINAL" in globals() and Path(VIDEO_ORIGINAL).is_file():
        audio = subprocess.run(
            ["ffprobe", "-v", "error", "-select_streams", "a",
             "-show_entries", "stream=index", "-of", "csv=p=0", str(VIDEO_ORIGINAL)],
            capture_output=True, text=True,
        ).stdout.strip()
        if audio:
            if max_frames > 0 and start_mode != "begin":
                print("Audio was not muxed because the clip starts in the middle.")
            else:
                muxed = out_dir / f"{clip}_cutout_with_audio.mov"
                try:
                    subprocess.run(
                        ["ffmpeg", "-y", "-loglevel", "error",
                         "-i", str(cutout), "-i", str(VIDEO_ORIGINAL),
                         "-map", "0:v:0", "-map", "1:a:0",
                         "-c:v", "copy", "-c:a", "pcm_s16le", "-shortest", str(muxed)],
                        check=True,
                    )
                except subprocess.CalledProcessError as exc:
                    print("Audio mux failed; keeping the video-only cutout.", exc)
                else:
                    print("Audio muxed:", muxed)
                    result = muxed
        else:
            print("The source has no audio track.")

    print("\nFull transparent video:", result, f"({human(result.stat().st_size)})")
    display(FileLink(str(result), result_html_prefix="Download transparent .mov: "))
    if preview_ok and preview.is_file():
        display(FileLink(str(preview), result_html_prefix="Download preview MP4: "))
    display(FileLink(str(still), result_html_prefix="Download frame 0 PNG: "))

    extras = []
    extras += sorted(out_dir.glob(f"{clip}_alpha.mp4"))
    extras += sorted(out_dir.glob(f"{clip}_fg.mp4"))
    extras += sorted(out_dir.glob(f"{clip}_checker_comp_despill*.mp4"))
    for extra in extras:
        print("Additional output:", extra.name, f"({human(extra.stat().st_size)})")
        display(FileLink(str(extra), result_html_prefix=f"Download {extra.name}: "))
else:
    print("No cutout output found; listing other outputs.")
    for p in sorted(out_dir.glob("*")):
        if p.is_file() and p.resolve() != still.resolve() and p.resolve() != preview.resolve():
            print(p.name, human(p.stat().st_size))
            display(FileLink(str(p), result_html_prefix=f"Download {p.name}: "))


<details>
<summary><b>⚙️ Parameter reference — what each advanced setting does</b></summary>

The transparent ProRes 4444 `cutout` is always written. Everything below is optional.

| Setting | Default | Effect |
|---|---|---|
| `save_alpha` | off | Also save the black/white matte as `..._alpha.mp4`. |
| `save_fg` | off | Also save the raw foreground RGB as `..._fg.mp4`. |
| `save_checker` | off | Also save a full-resolution foreground-over-checker preview. |
| `max_frames` | 0 | Number of frames to process. **0 = the entire video.** |
| `start_mode` | `begin` | Where to start when `max_frames > 0`: `begin`, `middle`, or `random_middle`. |
| `seed` | 0 | Deterministic seed for `random_middle`. |
| `hint_quality` | 0.95 | Trust in an uploaded hint. 0.95 = clean artist matte; 0.75 = rough; ≤0.5 = loose/scribble. |
| `carry_hint` | on | Reuse the model's own overlapping-window predictions as a continuing hint. Disable to treat a full hint video as authoritative on every pass. |
| `carry_hint_quality` | 0.95 | Trust in those carried hints. |
| `despill_strength` | 0.5 | Green/blue spill removal in the cutout. 0 = none, 1 = full. |
| `memory_mode` | Auto | Auto picks the largest Hann window that fits free VRAM. Force low VRAM stages features to disk. Manual exposes the controls below. |
| `hann_chunk` | 8 | Manual only: frames per stage-1 window. Larger = more temporal context, more VRAM. 80 is the maximum-context setting. |
| `hann_stride` | 4 | Manual only: frames between windows. 0 = half of `hann_chunk`. |
| `normalize_video` | off | Re-encode to H.264 before inference. Useful for exotic codecs/containers. Rotated phone videos are auto-normalized at upload time. |
| `downscale_720p` | off | Process a 720p copy for speed. The output is 720p. |
| `keep_audio` | on | Mux the original audio into a separate `..._cutout_with_audio.mov` when possible. |
| `cutout_linear` | off | Write straight scene-linear RGB instead of sRGB. For linear VFX compositing only. |

**Hints.** A matte is a black/white image: white means subject, black means background. An RGBA hint uses its alpha channel. A **first-frame matte** is usually the best practical mode: it gives the model a clear identity at the start and then tracks. A **full hint video** is the most controlled mode; it must match the processed resolution and cover the frames being processed.

**Hann windows.** Stage 1 combines overlapping temporal windows. Auto chooses from free GPU memory. If you set a manual window, auto selection is disabled. Rough guidance: `80/40` = max context/high VRAM; `48/24`, `24/12`, `8/4` = progressively lower memory; `1/1` = one frame at a time, lowest memory, no temporal context.

**Outputs.** `cutout` = despilled foreground with straight alpha (the transparent video). `alpha` = matte. `fg` = raw foreground RGB. `checker` = foreground composited over a checkerboard. The video path is display-referred and clipped to [0,1]; for unclipped HDR use the CLI with `--output_format exr`.

**What the notebook runs.** With defaults:

```bash
python infer.py \
  --input /content/work/<name>/<name>.mp4 \
  --output_dir /content/outputs/<name> \
  --outputs cutout \
  --hint_quality 0.95 \
  --carry_hint_quality 0.95 \
  --despill_strength 0.5
```

Native resolution is preserved; the model pads internally and crops back.

</details>


<details>
<summary><b>🛠️ Troubleshooting</b></summary>

**No GPU / “requires a CUDA GPU”.** In Colab: *Runtime → Change runtime type → Hardware accelerator → GPU*, then re-run the check cell.

**Colab asks to restart after installing PyTorch.** You can ignore it: inference runs in a fresh subprocess with the new PyTorch. If you do restart, re-run from the GPU check; setup skips finished work.

**CUDA out of memory.** Open the advanced settings and either set `memory_mode = Force low VRAM`, or `Manual Hann window` with `hann_chunk = 8`, `hann_stride = 4` (then 4/2). Also try `downscale_720p` or a smaller `max_frames`. Restart the runtime before retrying after a hard OOM.

**T4 / bfloat16.** Pre-Ampere GPUs (T4/Turing) lack native bfloat16. The setup cell automatically patches the two runtime files to use float16 and prints a notice. This is experimental but uses the same 2-byte precision; if you hit a CUDA error, switch to an A100/L4 runtime.

**The run is very slow.** Free T4 instances are slow for this model. Use an A100/L4 runtime, enable `downscale_720p`, or limit frames while testing. The first run also spends time loading the model.

**The session disconnected.** `/content` is erased when the session ends, including outputs and weights. Download results before disconnecting. The notebooks intentionally do not use Google Drive.

**The preview won’t play / looks like a normal MP4.** The inline preview is an 8-second H.264 transcode over a checkerboard. The real deliverable is the ProRes 4444 `.mov` with alpha, intended for editors/compositors. Import it into Premiere, Resolve, Final Cut, After Effects, etc.

**The subject is wrong or drifts.** Upload a first-frame matte in the optional section (white = subject, black = background), then re-run. For maximum control, provide a full hint video and disable `carry_hint`.

**Green/blue spill on edges.** Raise `despill_strength`. A cleaner hint also helps.

**My phone video is rotated.** The upload cell detects rotation metadata and creates an auto-rotated copy. If it still looks wrong, enable `normalize_video`.

**Full hint video rejected.** It must have the same resolution as the processed video and enough frames. Disable `downscale_720p`, or pre-scale the hint to the processed resolution.

**Log / ACES / HDR footage.** The runtime expects sRGB/Rec.709-like display-coded input; it does not convert Log/PQ/HLG/ACES automatically. Convert upstream. For linear EXR sequences, use the CLI with `--frame_dir_linear`.

**Audio is missing from the output.** The model writes video only. The notebook muxes audio into `_cutout_with_audio.mov` when the source has an audio track. If you trimmed from the middle (`start_mode`), audio is not muxed automatically.

**`prores_ks` / ffmpeg missing.** The setup cell installs/checks ffmpeg and aborts with instructions if the encoder is unavailable. Re-run: `apt-get install -y ffmpeg`.

</details>


<details>
<summary><b>📜 License & acknowledgements</b></summary>

CorridorKey 2.0 was designed and trained by **Zachary Harris**, in direct collaboration with **Niko** and **Corridor Digital**.

- You may use CorridorKey and its outputs for personal or commercial work. What you make is yours.
- You may not sell the tool, its code, or its model, or offer it as a paid API/inference service, without permission from Corridor Digital.
- If you build on the code or model, your release must be freely accessible, stay under the same license, and keep the CorridorKey name. Donationware/pay-what-you-want is fine.
- Organizations with more than US $20 million in annual revenue need to purchase a CorridorKey plugin seat for each person using the tool.
- Commercial software integrations: contact contact@corridordigital.com.

Full terms: [LICENSE](https://github.com/nikopueringer/CorridorKey2.0/blob/master/LICENSE). Official plugin/app: [corridorkey.com](https://corridorkey.com).

CorridorKey builds on three frozen foundation models:

- **C-RADIO** by NVIDIA — NVIDIA Open Model License
- **MoGe** by Microsoft — MIT
- **RVM** by Google DeepMind — code Apache 2.0, weights CC-BY 4.0

Details are in [THIRD_PARTY_NOTICES.md](https://github.com/nikopueringer/CorridorKey2.0/blob/master/THIRD_PARTY_NOTICES.md).

</details>
